# Plot precipitation efficiency and related metrics from the idealized square-domain RCE (SAM) model output.

This contains code to produce one of the main paper figures, along with several that are included in the paper's supporting information.

## Main settings

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import pickle
from scipy import stats
from scipy.stats import mannwhitneyu
from statannotations.Annotator import Annotator

In [ ]:
data_main = './data/'
figdir = './figures/'

# RCE square domain sims:
#   control: 24 3D files per day, but we read/retain only 4/day
#   radhomo: 4 3D files per day
# 

# 300 # seconds per step (= 5 min)
# 300*24 # label numbers

# 3D output is
#   24/day for RCE
#   4/day for RCE_RH

# Processing matches these by keeping only 4/day for RCE

test_names = [
    'RCE_CTL',
    'RCE_OffCRF',
    ]

### Read coordinates

In [ ]:
# Read/write domain variables, time arrays to pickle
pickle_file_exper_settings = data_main+'exper_coords_RCE.pickle'

# Read from pickle
with open(pickle_file_exper_settings, 'rb') as f:
    pickle_in = pickle.load(f)

x = pickle_in['x']
y = pickle_in['y']
z = pickle_in['z']
nx = pickle_in['nx']
ny = pickle_in['ny']
nz = pickle_in['nz']
times_3d = pickle_in['times_3d']
files_3d = pickle_in['files_3d']
times_3d_rh = pickle_in['times_3d_rh']
files_3d_rh = pickle_in['files_3d_rh']

# Apply analysis day bounds to times and files
day_bounds = (1,100)
times_3d = times_3d[(times_3d >= day_bounds[0]) & (times_3d <= day_bounds[1])]
files_3d = [f for t, f in zip(times_3d, files_3d) if t >= day_bounds[0] and t <= day_bounds[1]]

## Plot functions

In [ ]:
def read_tser_pickle(filename):
    with open(filename, 'rb') as f:
        var = pickle.load(f)
    return var

In [ ]:
# Function to compute running mean
def running_mean_conf(time_series):
    nd_smooth    = 5 # days
    ntpday = 4 # timesteps per day
    window_size = nd_smooth*ntpday  # Adjust as needed
    tser_smooth = np.convolve(time_series, np.ones(window_size) / window_size, mode='valid')
    # Compute standard error of the mean
    standard_error = stats.sem(time_series)  # Standard error of the original data
    z_score = 1.96 # Z-score for 95% confidence interval
    confidence_interval = z_score * standard_error  # 95% confidence interval
    return tser_smooth, confidence_interval, window_size

## Plots

In [ ]:
colors_paper = ['black', 'dodgerblue']
sns.set_theme(style="ticks", rc={'xtick.bottom': True, 'ytick.left': True},
              font_scale=1.3)#, "axes.spines.right": False, "axes.spines.top": False})
sns.set_palette('muted')

def plot_tser(ax, ivar, ivar_rh, test_names):
    for (pltvar, legend) in zip([ivar, ivar_rh], test_names[0:2]):
        ivar_smooth, confidence_interval, nwindow = running_mean_conf(pltvar)
        x_smoothed = times_3d[nwindow//2:-nwindow//2+1]
        ax.plot(times_3d, pltvar, linewidth=0.4, color=colors_paper[test_names.index(legend)], alpha=0.3)
        ax.plot(x_smoothed, ivar_smooth, label=legend, linewidth=1.8, zorder=1, color=colors_paper[test_names.index(legend)])
    return None

def boxplot_setup(ivars_in, d_sel, times_3d):
    ivar_sav = []
    for ivar in ivars_in:
        time_str = str(int(times_3d[d_sel[0]]))+'-'+str(int(times_3d[d_sel[1]]))+' d'
        ivar_sav.append(ivar[d_sel[0]:d_sel[1]])
    mannw_stat, p_value = mannwhitneyu(ivar_sav[0],
                                ivar_sav[1],
                                method='asymptotic',
                                )
    return ivar_sav, time_str, p_value

def plot_boxplot(ax, ivars):
    colors = sns.color_palette()[:2]
    sns.violinplot(ivars,
                width=0.7,
                palette=colors, alpha=0.8,
                inner="box", split=True,
                # density_norm='area',
                gap=0.01,
                ax=ax)
    # Annotate with significance
    pairs = [(0, 1)]
    annotator = Annotator(ax, pairs, data=ivars)
    annotator.configure(test='Mann-Whitney', text_format='star', loc='outside', verbose=0)
    annotator.apply_and_annotate()
    return None

In [ ]:
from matplotlib import patches

def calc_percent_difference(ivar_ctl, ivar_rh):
    ivar_ctl = np.asarray(ivar_ctl, dtype=float)
    ivar_rh = np.asarray(ivar_rh, dtype=float)
    pct_diff = np.full_like(ivar_ctl, np.nan, dtype=float)
    valid = ~np.isclose(ivar_ctl, 0.0)
    pct_diff[valid] = 100.0 * (ivar_rh[valid] - ivar_ctl[valid]) / ivar_ctl[valid]
    return pct_diff

def plot_percent_diff_tser(ax, ivar_ctl, ivar_rh, legend='RCE_RH vs RCE_CTL'):
    ctl_smooth, _, nwindow = running_mean_conf(ivar_ctl)
    rh_smooth, _, nwindow = running_mean_conf(ivar_rh)
    x_smoothed = times_3d[nwindow//2:-nwindow//2+1]
    pct_diff = calc_percent_difference(ctl_smooth, rh_smooth)
    ax.plot(x_smoothed, pct_diff, label=legend, linewidth=2.0, zorder=1, color=colors_paper[1])
    return pct_diff

def annotate_percent_diff(ax, pct_diff, d_sel, yloc=0.03):
    pct_subset = pct_diff[d_sel[0]:d_sel[1]]
    pct_mean = np.nanmean(pct_subset)
    time_str = f"{int(times_3d[d_sel[0]])}-{int(times_3d[d_sel[1]])} d"
    ax.text(0.03, yloc, f"mean ({time_str}) = {pct_mean:.1f}%", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
    return None

def add_time_window_box(ax, d_sel, y_norm_bottom=0.15, y_norm_height=0.4):
    y_data_bottom = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom)))[1]
    y_data_top = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom + y_norm_height)))[1]
    y_data_height = y_data_top - y_data_bottom
    x_data = times_3d[d_sel[0]]
    x_width = times_3d[d_sel[1]] - times_3d[d_sel[0]]
    box = patches.Rectangle((x_data, y_data_bottom), x_width, y_data_height,
                            fill=False, edgecolor='black', linewidth=1)
    ax.add_patch(box)
    return None

### PE, CRH, size

In [ ]:
from matplotlib import patches

def combined_plot():

    #### PLOT SETUP ########################

    fig_x, fig_y = 12, 3
    ncols=3
    nrows = 1
    fig, axs = plt.subplots(nrows, ncols, #width_ratios=[tser_ratio, box_ratio, tser_ratio, box_ratio, tser_ratio, box_ratio],
                            figsize=(fig_x,fig_y), layout='constrained', squeeze=True, dpi=300)
    # plt.suptitle(title)

    # Clean up time series subplots
    axs[1].set_xlabel('Time [d]')
    for irow in range(ncols):
        axs[irow].set_xlim(-1, 60)
        sns.despine(offset=10,ax=axs[irow], left=False, right=True, bottom=False)

    # Time subset for boxplots
    d_sel = (0, 20*4)

    def add_text_ax0(ax, p_value):
        # xloc, yloc = 0.95, 0.01
        xloc, yloc = 0.03, 0.03
        # ax.text(xloc, yloc, f"$p$ ({time_str}) = {str(np.round(p_value, 3))}", ha='right', va='bottom', transform=ax.transAxes, fontsize=12)
        ax.text(xloc, yloc, f"$p$ ({time_str}) = {str(np.round(p_value, 3))}", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
        return None
    def add_text(ax, p_value):
        # xloc, yloc = 0.95, 0.01
        xloc, yloc = 0.03, 0.03
        # ax.text(xloc, yloc, f"{str(np.round(p_value, 3))}", ha='right', va='bottom', transform=ax.transAxes, fontsize=12)
        ax.text(xloc, yloc, f"{str(np.round(p_value, 3))}", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
        return None

    #### PE ########################

    irow = 0
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}pe_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}pe_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min = -0.02
    y_max = 0.4
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    ax_tser.set_title(r'(a) $\epsilon$')
    ax_tser.set_ylabel('-')

    # Plot line plot with PE for both tests
    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text_ax0(ax_boxplot, p_value)


    #### RELATIVE HUM ########################

    irow = 1
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}crh_mid_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}crh_mid_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    # y_min, y_max = axs[1,0].get_ylim()
    y_min, y_max = 71, 90
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    title = r'(b) $r_{mid}$'
    ax_tser.set_title(title)
    ax_tser.set_ylabel('%')

    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text(ax_boxplot, p_value)


    #### DeepC Size ########################

    irow = 2
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}deepc_sizes_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}deepc_sizes_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min, y_max = 5.1, 7.5
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    title = '(c) Mean size'
    ax_tser.set_title(title)
    ax_tser.set_ylabel('km')

    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text(ax_boxplot, p_value)


    ############################

    # Add legend
    axs[0].legend(frameon=False, loc='upper left')#, bbox_to_anchor=(0.05, 1.05))

    # Annotate a box into each panel
    # Define box position in normalized y-coordinates (consistent across panels)
    y_norm_bottom = 0.15  # 5% from bottom of axis
    y_norm_height = 0.4   # 20% of axis height

    for ax in axs:
        # Convert normalized y to data coordinates
        y_data_bottom = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom)))[1]
        y_data_top = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom + y_norm_height)))[1]
        y_data_height = y_data_top - y_data_bottom

        # x in data coordinates (same across all panels)
        x_data = times_3d[d_sel[0]]
        x_width = times_3d[d_sel[1]] - times_3d[d_sel[0]]

        box = patches.Rectangle((x_data, y_data_bottom), x_width, y_data_height,
                                fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(box)

    # Save figure
    # plt.savefig(figdir+'rce_combined_plot.png',dpi=400, facecolor='white', \
    #     bbox_inches='tight')#, pad_inches=0.2)
    plt.savefig(figdir+'rce_combined_plot.pdf', bbox_inches='tight')
    plt.show()
    plt.close()
    return None

combined_plot()

### PE, M_u, M_d

In [ ]:
def combined_plot():

    #### PLOT SETUP ########################

    # fig_x, fig_y = 4, 6.5
    fig_x, fig_y = 12, 3
    ncols=3
    nrows = 1
    fig, axs = plt.subplots(nrows, ncols, #width_ratios=[tser_ratio, box_ratio, tser_ratio, box_ratio, tser_ratio, box_ratio],
                            figsize=(fig_x,fig_y), layout='constrained', squeeze=True, dpi=300)
    # plt.suptitle(title)

    # Clean up time series subplots
    axs[1].set_xlabel('Time [d]')
    for irow in range(ncols):
        axs[irow].set_xlim(-1, 60)
        sns.despine(offset=10,ax=axs[irow], left=False, right=True, bottom=False)

    # Time subset for boxplots
    d_sel = (0, 20*4)

    def add_text_ax0(ax, p_value, xloc=0.03, yloc=0.03):
        ax.text(xloc, yloc, f"$p$ ({time_str}) = {str(np.round(p_value, 3))}", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
        return None
    def add_text(ax, p_value, xloc=0.03, yloc=0.03):
        ax.text(xloc, yloc, f"{str(np.round(p_value, 3))}", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
        return None


    #### PE ########################

    irow = 0
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}pe_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}pe_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min = -0.02
    y_max = 0.4
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    ax_tser.set_title(r'(a) $\epsilon$')# ('+f'{pclass_names[ipclass]})')
    ax_tser.set_ylabel('-')

    # Plot line plot with PE for both tests
    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text_ax0(ax_boxplot, p_value)


    #### M_u ########################

    irow = 1
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}mu_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}mu_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min, y_max = 1000, 6900
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    title = rf'(b) $M_u$'
    ax_tser.set_title(title)
    ax_tser.set_ylabel('kg/m/s')

    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text(ax_boxplot, p_value, yloc=0.27)


    #### M_d ########################

    irow = 2
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}md_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}md_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min, y_max = 1000, 5500
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    title = rf'(c) $M_d$'# ('+f'{pclass_names[ipclass]})'
    ax_tser.set_title(title)
    ax_tser.set_ylabel('kg/m/s')

    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text(ax_boxplot, p_value, yloc=0.27)


    ############################

    # Add legend
    axs[0].legend(frameon=False, loc='upper left')#, bbox_to_anchor=(0.05, 1.05))

    # Annotate a box into each panel
    # Define box position in normalized y-coordinates (consistent across panels)
    y_norm_bottom = 0.15  # 5% from bottom of axis
    y_norm_height = 0.4   # 20% of axis height

    for iax, ax in enumerate(axs):
        if iax > 0:
            y_norm_bottom = 0.38  # 5% from bottom of axis

        # Convert normalized y to data coordinates
        y_data_bottom = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom)))[1]
        y_data_top = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom + y_norm_height)))[1]
        y_data_height = y_data_top - y_data_bottom

        # x in data coordinates (same across all panels)
        x_data = times_3d[d_sel[0]]
        x_width = times_3d[d_sel[1]] - times_3d[d_sel[0]]

        box = patches.Rectangle((x_data, y_data_bottom), x_width, y_data_height,
                                fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(box)

    # Save figure
    # plt.savefig(figdir+'rce_vmf.png',dpi=400, facecolor='white', \
    #     bbox_inches='tight')#, pad_inches=0.2)
    plt.savefig(figdir+'rce_vmf.pdf', bbox_inches='tight')
    plt.show()
    plt.close()
    return None

combined_plot()

### Percent difference: PE, CRH, size

In [ ]:
def set_percent_diff_ylim(ax, pct_diff, min_pad=2.0):
    finite = pct_diff[np.isfinite(pct_diff)]
    if finite.size == 0:
        ax.set_ylim(-10, 10)
        return None
    y_min, y_max = np.nanpercentile(finite, [2, 98])
    spread = y_max - y_min
    pad = max(min_pad, 0.15 * spread if spread > 0 else min_pad)
    ax.set_ylim(y_min - pad, y_max + pad)
    return None

def combined_plot_percent_diff_pe_crh_size():

    fig_x, fig_y = 12, 3
    ncols = 3
    fig, axs = plt.subplots(1, ncols, figsize=(fig_x, fig_y), layout='constrained', squeeze=True, dpi=300)

    axs[1].set_xlabel('Time [d]')
    for iax, ax in enumerate(axs):
        ax.set_xlim(-1, 60)
        if iax == 0:
            ax.set_ylabel('% change from CTL')
        sns.despine(offset=10, ax=ax, left=False, right=True, bottom=False)

    d_sel = (0, 20 * 4)
    var_specs = [
        ('pe', r'(a) $\epsilon$'),
        ('crh_mid', r'(b) $r_{mid}$'),
        ('deepc_sizes', '(c) Mean size'),
    ]

    for iax, (var_name, title) in enumerate(var_specs):
        ax = axs[iax]
        ivar_ctl = read_tser_pickle(f"{data_main}{var_name}_{test_names[0]}.pkl")
        ivar_rh = read_tser_pickle(f"{data_main}{var_name}_{test_names[1]}.pkl")
        pct_diff = plot_percent_diff_tser(ax, ivar_ctl, ivar_rh)
        # set_percent_diff_ylim(ax, pct_diff)
        ax.set_title(title)
        # annotate_percent_diff(ax, pct_diff, d_sel)
        # add_time_window_box(ax, d_sel)
        # Add zero line
        ax.axhline(0, color='black', linestyle='-', zorder=0)

    axs[0].set_ylim(-60, 10)
    axs[1].set_ylim(-13, 2)
    axs[2].set_ylim(-21, 2)

    # axs[1].legend(frameon=False, loc='lower left')
    plt.savefig(figdir + 'rce_combined_plot_pctdiff.pdf', bbox_inches='tight')
    plt.show()
    plt.close()
    return None

combined_plot_percent_diff_pe_crh_size()

### Cluster mean area and mid-level CRH by class

In [ ]:
mean_cluster_size_data = read_tser_pickle(
    data_main + 'RCE_mean_cluster_size_percent_difference.pickle'
)
mean_midcrh_data = read_tser_pickle(
    data_main + 'RCE_mean_midcrh_percent_difference.pickle'
)

cluster_labels = ['DeepC', 'Strat', 'Anvil', 'DSA']
all_labels = ['NonCloud', 'DeepC', 'Strat', 'Anvil', 'DSA']

# class_colors = dict(zip(
#     all_labels, sns.color_palette('colorblind', n_colors=len(all_labels))
# ))
# class_colors = dict(zip(
#     all_labels, sns.color_palette('muted', n_colors=len(all_labels))
# ))
# Replace the first color with black for NonCloud and shift colors up in the palette
class_colors = {}
class_colors['NonCloud'] = 'black'
for ilabel in all_labels[1:]:
    class_colors[ilabel] = sns.color_palette('muted', n_colors=len(all_labels))[all_labels.index(ilabel)-1]

fig, axs = plt.subplots(1, 2, figsize=(9, 3),
                        layout='constrained', dpi=300)

legend_handles = {}
for label in all_labels:
    class_series = mean_midcrh_data['series'][label]
    line, = axs[0].plot(
        class_series['time_smoothed_days'],
        class_series['percent_difference_from_ctl_smoothed'],
        color=class_colors[label], label=label, linewidth=1.8, zorder=1,
    )
    legend_handles[label] = line

for label in cluster_labels:
    class_series = mean_cluster_size_data['series'][label]
    axs[1].plot(
        class_series['time_smoothed_days'],
        class_series['percent_difference_from_ctl_smoothed'],
        color=class_colors[label], label=label, linewidth=1.8, zorder=1,
    )

for ax in axs:
    ax.axhline(0, color='0.5', linewidth=0.8, zorder=0)
    ax.set_xlim(-1, 80)
    ax.set_xlabel('Time [d]')
    sns.despine(offset=10, ax=ax, left=False, right=True, bottom=False)

axs[0].set_title(r'(a) Mean $r_{mid}$')
axs[0].set_ylim(-13, 30)
axs[0].set_ylabel('% change from CTL')

axs[1].set_title('(b) Mean size')
axs[1].set_ylim(-30, 30)
axs[0].legend(
    [legend_handles[label] for label in all_labels], all_labels,
    frameon=False, loc='upper right', fontsize=9,
)

plt.savefig(
    figdir + 'rce_cluster_area_midcrh_pctdiff_pclasses.pdf',
    bbox_inches='tight',
)
plt.show()
plt.close()

### Domain area fraction

In [ ]:
import xarray as xr

pclass_area_file = data_main + 'rce_pclass_diagnostics_timeseries.nc'
with xr.open_dataset(pclass_area_file) as ds:
    expected_experiments = ['RCE_CTL', 'RCE_RH']
    np.testing.assert_array_equal(ds['experiment'].values, expected_experiments)
    np.testing.assert_array_equal(ds['pclass_name'].values, cluster_labels)
    assert ds.sizes['time'] == 397
    pclass_area_percent = (100.0 * ds['pclass_area_fraction']).load()
    pclass_area_time = ds['time'].values.copy()

experiment_specs = [
    ('RCE_CTL', '(a) CTL'),
    ('RCE_RH', '(b) OffCRF'),
]

fig, axs = plt.subplots(
    1, 3, figsize=(9, 2.7), layout='constrained', dpi=300,
)
axs[1].sharey(axs[0])

plot_labels = cluster_labels[0:3]
legend_handles = []
area_smoothed = {}
for iexperiment, (experiment, title) in enumerate(experiment_specs):
    ax = axs[iexperiment]
    for ipclass, label in enumerate(plot_labels):
        class_area = (
            pclass_area_percent.sel(experiment=experiment)
            .isel(pclass=ipclass)
            .values
        )
        class_area_smoothed, _, nwindow = running_mean_conf(class_area)
        time_smoothed = pclass_area_time[nwindow // 2:-nwindow // 2 + 1]
        assert class_area_smoothed.size == time_smoothed.size == 378
        assert np.isfinite(class_area_smoothed).all()
        area_smoothed[(experiment, label)] = class_area_smoothed
        line, = ax.plot(
            time_smoothed, class_area_smoothed,
            color=class_colors[label], label=label,
            linewidth=1.4, zorder=1,
        )
        if iexperiment == 0:
            legend_handles.append(line)

    ax.set_title(title)
    ax.set_xlim(0, 80)
    ax.set_ylim(0, 2.6)
    ax.set_xlabel('Time [d]')
    sns.despine(offset=10, ax=ax, left=False, right=True, bottom=False)

axs[1].tick_params(labelleft=False)

ax = axs[2]
for label in plot_labels:
    area_percent_difference = calc_percent_difference(
        area_smoothed[('RCE_CTL', label)],
        area_smoothed[('RCE_RH', label)],
    )
    assert area_percent_difference.size == time_smoothed.size == 378
    assert np.isfinite(area_percent_difference).all()
    ax.plot(
        time_smoothed, area_percent_difference,
        color=class_colors[label], label=label,
        linewidth=1.4, zorder=1,
    )

ax.axhline(0, color='0.5', linewidth=0.8, zorder=0)
ax.set_title('(c) Difference')
ax.set_xlim(0, 80)
ax.set_ylim(-50, 200)
ax.set_xlabel('Time [d]')
ax.set_ylabel('% change from CTL')
sns.despine(offset=10, ax=ax, left=False, right=True, bottom=False)

axs[0].set_ylabel('Area fraction [%]')
fig.legend(
    legend_handles, plot_labels, frameon=False,
    loc='outside upper center', ncol=3, fontsize=10,
)

plt.savefig(
    figdir + 'rce_pclass_area_fraction_timeseries.pdf',
    bbox_inches='tight',
)
plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3), layout='constrained', dpi=300)

for ipclass, label in enumerate(cluster_labels[0:3]):
    ctl_area = (
        pclass_area_percent.sel(experiment='RCE_CTL')
        .isel(pclass=ipclass)
        .values
    )
    offcrf_area = (
        pclass_area_percent.sel(experiment='RCE_RH')
        .isel(pclass=ipclass)
        .values
    )
    ctl_area_smoothed, _, nwindow = running_mean_conf(ctl_area)
    offcrf_area_smoothed, _, _ = running_mean_conf(offcrf_area)
    time_smoothed = pclass_area_time[nwindow // 2:-nwindow // 2 + 1]
    area_percent_difference = calc_percent_difference(
        ctl_area_smoothed, offcrf_area_smoothed,
    )
    assert area_percent_difference.size == time_smoothed.size == 378
    assert np.isfinite(area_percent_difference).all()
    ax.plot(
        time_smoothed, area_percent_difference,
        color=class_colors[label], label=label,
        linewidth=1.8, zorder=1,
    )

ax.axhline(0, color='0.5', linewidth=0.8, zorder=0)
ax.set_title('OffCRF class domain area')
ax.set_xlim(0, 100)
# ax.set_yscale('symlog', linthresh=10)
# ax.set_ylim(-30, 600)
ax.set_ylim(-50, 200)
ax.set_xlabel('Time [d]')
ax.set_ylabel('% change from CTL')
fig.legend(
    loc='outside upper center', ncol=4, frameon=False, fontsize=9,
)
sns.despine(offset=10, ax=ax, left=False, right=True, bottom=False)

plt.savefig(
    figdir + 'rce_pclass_area_fraction_percent_difference.pdf',
    bbox_inches='tight',
)
plt.show()
plt.close()